# Part C — Support Ticket Triage with Guardrails

This notebook builds a privacy-safe and resilient support ticket triage pipeline using the supplied offline LLM.

It applies input guardrails, redacts personal information, validates model responses, handles transient failures, tracks token usage, and routes tickets using customer segments from Part B.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import random
import time
from pathlib import Path
from collections import Counter

### Define Project Paths

Define the project root directory and the locations of the raw input data and output files.

All generated Part C results will be saved in the project's outputs folder.

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
OUTPUTS = PROJECT_ROOT / "outputs"

OUTPUTS.mkdir(parents=True, exist_ok=True)

### Load Support Tickets and Customer Segments

Load the original support tickets and customer segment output generated in Part B.

Use `keep_default_na=False` when reading tickets so empty ticket text remains an empty string. This allows us to identify empty tickets correctly during input validation.

In [3]:
tickets = pd.read_csv(
    RAW_DATA / "tickets.csv",
    keep_default_na=False
)

customer_segments = pd.read_csv(
    OUTPUTS / "customer_segments.csv"
)

print("Tickets shape:", tickets.shape)
print("Customer segments shape:", customer_segments.shape)

print("\nTicket columns:")
print(tickets.columns.tolist())

print("\nCustomer segment columns:")
print(customer_segments.columns.tolist())

Tickets shape: (164, 5)
Customer segments shape: (1175, 7)

Ticket columns:
['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']

Customer segment columns:
['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']


### Inspect the Supplied Mock LLM

Before building the triage pipeline, inspect the supplied `mock_llm.py` implementation.

Understand its classification function, response format, token usage information, retry-related exception, and reset functionality. We will use its existing interface rather than creating a replacement.

In [4]:
mock_path = PROJECT_ROOT / "mock_llm.py"

print(mock_path.read_text(encoding="utf-8"))

"""mock_llm.py -- deterministic, offline stand-in for a hosted LLM chat API (no key, no network, no cost).
Interface mirrors a hosted chat API:  response = chat(messages, tools=None)
    response.choices[0].message.content / .tool_calls ; response.choices[0].finish_reason ; response.usage
Three behaviours are selected by a marker word in the FIRST (system) message:
    "CLASSIFY"  -> support-ticket classifier       "POLICY ASSISTANT" -> grounded policy answerer
    "AGENT"     -> refund agent that requests tools (ReAct style)"""
import json, math, re
from types import SimpleNamespace as NS

class TransientLLMError(Exception):
    """Simulated temporary failure (timeout / server busy). Retrying the same call succeeds."""

_state = {"calls": 0, "ok": 0}      # call counters that drive the simulated misbehaviour

def reset():
    """Restart the counters. Call once before each independent batch run."""
    _state["calls"] = 0; _state["ok"] = 0

CATS = ["billing", "delivery", "technical", "

### Import the Offline LLM

Import the supplied offline mock LLM module.

This module simulates support-ticket classification without requiring an API key, internet connection, or external model service. It also provides token usage information and simulated transient errors for testing retry handling.

In [5]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

import mock_llm

print("Offline mock LLM imported successfully.")
print("Maximum input tokens:", 400)

Offline mock LLM imported successfully.
Maximum input tokens: 400


### Apply Input Guardrails

Estimate the token count of each ticket using the supplied mock LLM's `est_tokens()` function.

Skip empty tickets and flag tickets exceeding the 400-token limit. Only eligible tickets will proceed to PII redaction and model classification.

In [6]:
MAX_INPUT_TOKENS = 400

ticket_work = tickets.copy()

ticket_work["estimated_tokens"] = ticket_work["text"].apply(
    mock_llm.est_tokens
)

ticket_work["status"] = "eligible"

empty_mask = ticket_work["text"].str.strip().eq("")
long_mask = (
    ~empty_mask
    & (ticket_work["estimated_tokens"] > MAX_INPUT_TOKENS)
)

ticket_work.loc[empty_mask, "status"] = "skipped_empty"
ticket_work.loc[long_mask, "status"] = "flagged_long"

print("Total tickets:", len(ticket_work))
print("\nInput guardrail results:")
print(ticket_work["status"].value_counts())

print(
    "\nMaximum estimated tokens among long tickets:",
    ticket_work.loc[long_mask, "estimated_tokens"].max()
)

print(
    "\nEligible tickets for classification:",
    (ticket_work["status"] == "eligible").sum()
)

Total tickets: 164

Input guardrail results:
status
eligible         156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64

Maximum estimated tokens among long tickets: 734

Eligible tickets for classification: 156


### Redact Personally Identifiable Information

Before sending eligible ticket messages to the model, redact the following information using the prescribed regular expressions:

- Email addresses → `[EMAIL]`
- Indian mobile numbers → `[PHONE]`
- Order IDs → `[ORDER_ID]`

Apply the replacements in the required order. Count substitutions across all tickets, including tickets skipped by the input guardrails, and verify that no matching PII remains after redaction.

In [7]:
EMAIL_PATTERN = r"[\w.+-]+@[\w-]+\.[\w.]+"
PHONE_PATTERN = r"\b[6-9]\d{9}\b"
ORDER_PATTERN = r"ORD-\d{6}"

def redact_pii(text):
    counts = {
        "email": 0,
        "phone": 0,
        "order_id": 0
    }

    text, counts["email"] = re.subn(
        EMAIL_PATTERN, "[EMAIL]", text
    )

    text, counts["phone"] = re.subn(
        PHONE_PATTERN, "[PHONE]", text
    )

    text, counts["order_id"] = re.subn(
        ORDER_PATTERN, "[ORDER_ID]", text
    )

    return text, counts


redacted_texts = []
redaction_counts = Counter()

for text in ticket_work["text"]:
    redacted, counts = redact_pii(text)
    redacted_texts.append(redacted)
    redaction_counts.update(counts)

ticket_work["redacted_text"] = redacted_texts

print("PII substitution counts:")
print("Emails:", redaction_counts["email"])
print("Phone numbers:", redaction_counts["phone"])
print("Order IDs:", redaction_counts["order_id"])
print("Total substitutions:", sum(redaction_counts.values()))

remaining_pii = {
    "email": ticket_work["redacted_text"].str.contains(
        EMAIL_PATTERN, regex=True
    ).sum(),
    "phone": ticket_work["redacted_text"].str.contains(
        PHONE_PATTERN, regex=True
    ).sum(),
    "order_id": ticket_work["redacted_text"].str.contains(
        ORDER_PATTERN, regex=True
    ).sum()
}

print("\nRemaining PII matches after redaction:")
print(remaining_pii)

assert sum(remaining_pii.values()) == 0
print("\nPII validation passed.")

PII substitution counts:
Emails: 55
Phone numbers: 91
Order IDs: 132
Total substitutions: 278

Remaining PII matches after redaction:
{'email': np.int64(0), 'phone': np.int64(0), 'order_id': np.int64(0)}

PII validation passed.


### Define Classification Prompts

Create two system prompts for support-ticket classification:

1. Zero-shot prompt: provides classification instructions without examples.
2. Few-shot prompt: provides labelled examples to guide classification.

Both prompts must contain the `CLASSIFY` marker, restrict categories to billing, delivery, technical, and fraud, and require a JSON-only response with `category` and `reason`.

The ticket's redacted text will be sent separately as the user message.

In [8]:
CATEGORIES = ["billing", "delivery", "technical", "fraud"]

zero_shot_prompt = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

Return JSON only in this exact structure:
{"category": "<category>", "reason": "<short reason>"}

Do not include explanations outside the JSON object.
"""

few_shot_prompt = """
CLASSIFY the customer support ticket into exactly one category.

Allowed categories:
- billing
- delivery
- technical
- fraud

EXAMPLES:
Ticket: I was charged twice for my purchase.
Response: {"category": "billing", "reason": "duplicate charge"}

Ticket: My account was hacked and I noticed unauthorized activity.
Response: {"category": "fraud", "reason": "unauthorized account activity"}

Ticket: My order has not arrived yet.
Response: {"category": "delivery", "reason": "order not received"}

Return JSON only in this exact structure:
{"category": "<category>", "reason": "<short reason>"}

Do not include explanations outside the JSON object.
"""

assert "CLASSIFY" in zero_shot_prompt
assert "CLASSIFY" in few_shot_prompt
assert "EXAMPLES:" not in zero_shot_prompt
assert "EXAMPLES:" in few_shot_prompt

for category in CATEGORIES:
    assert category in zero_shot_prompt
    assert category in few_shot_prompt

print("Zero-shot prompt validation: PASSED")
print("Few-shot prompt validation: PASSED")
print("Allowed categories:", CATEGORIES)
few_shot_examples = few_shot_prompt.split("EXAMPLES:", 1)[1].count("Ticket:")

print("Few-shot examples:", few_shot_examples)

Zero-shot prompt validation: PASSED
Few-shot prompt validation: PASSED
Allowed categories: ['billing', 'delivery', 'technical', 'fraud']
Few-shot examples: 3


### Validate Classification Responses

Validate every model response before accepting its classification.

- Valid JSON with an allowed category → `classified_json`.
- Non-JSON text containing exactly one allowed category word → `classified_keyword`.
- Invalid JSON categories, multiple category matches, or unrecognised responses → `flagged_invalid_output`.

Never guess or automatically correct an invalid category. Such tickets will be routed for human review.

In [9]:
def validate_response(response_text):
    try:
        parsed = json.loads(response_text)

        if (
            isinstance(parsed, dict)
            and parsed.get("category") in CATEGORIES
        ):
            return parsed["category"], "classified_json"

        return None, "flagged_invalid_output"

    except (json.JSONDecodeError, TypeError):
        words = re.findall(r"[a-z]+", str(response_text).lower())

        matches = set(words) & set(CATEGORIES)

        if len(matches) == 1:
            return matches.pop(), "classified_keyword"

        return None, "flagged_invalid_output"


# Test the three response-validation cases.
test_responses = [
    '{"category": "billing", "reason": "duplicate charge"}',
    'The customer appears to have a fraud-related problem.',
    '{"category": "shipping", "reason": "parcel movement"}'
]

for response in test_responses:
    print(response)
    print("Result:", validate_response(response))
    print()

{"category": "billing", "reason": "duplicate charge"}
Result: ('billing', 'classified_json')

The customer appears to have a fraud-related problem.
Result: ('fraud', 'classified_keyword')

{"category": "shipping", "reason": "parcel movement"}
Result: (None, 'flagged_invalid_output')



### Implement Retry Handling

Create a classification function that sends the system prompt and redacted ticket text to the offline LLM.

The function will:
- Verify that no email, phone number, or order ID remains in the user message.
- Retry transient errors up to three times after the initial attempt.
- Apply exponential backoff with random jitter.
- Log each retry with the ticket ID, retry number, and wait duration.
- Validate the response and return its category, status, token usage, and retry count.

In [10]:
BASE_DELAY = 0.05
MAX_RETRIES = 3

def classify_ticket(ticket_id, redacted_text, system_prompt):
    # Ensure the user message contains no unredacted PII.
    assert not re.search(EMAIL_PATTERN, redacted_text)
    assert not re.search(PHONE_PATTERN, redacted_text)
    assert not re.search(ORDER_PATTERN, redacted_text)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": redacted_text}
    ]

    retry_count = 0

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = mock_llm.chat(messages)

            response_text = response.choices[0].message.content
            category, status = validate_response(response_text)
            tokens_used = response.usage.total_tokens

            return {
                "category": category,
                "status": status,
                "tokens_used": tokens_used,
                "retry_count": retry_count
            }

        except mock_llm.TransientLLMError:
            if attempt == MAX_RETRIES:
                raise

            wait_time = (
                BASE_DELAY
                * (2 ** attempt)
                * random.uniform(0.9, 1.15)
            )

            retry_count += 1

            print(
                f"Ticket {ticket_id} | "
                f"Retry {retry_count}/{MAX_RETRIES} | "
                f"Waiting {wait_time:.4f} seconds"
            )

            time.sleep(wait_time)

### Execute Zero-Shot Classification

Run the complete ticket-processing pipeline using the zero-shot prompt.

Process tickets in their original file order. Skip tickets already stopped by the input guardrails, classify eligible tickets, and record each ticket's classification status, token usage, and retry count.

Reset the mock LLM immediately before the run so its simulated failures and responses start from a clean state.

In [11]:
def run_triage(system_prompt, token_budget=None):
    mock_llm.reset()

    results = []
    total_tokens = 0
    total_retries = 0

    for _, ticket in ticket_work.iterrows():
        ticket_id = ticket["ticket_id"]

        result = {
            "ticket_id": ticket_id,
            "customer_id": ticket["customer_id"],
            "true_category": ticket["true_category"],
            "category": None,
            "status": ticket["status"],
            "tokens_used": 0,
            "retry_count": 0
        }

        if ticket["status"] != "eligible":
            results.append(result)
            continue

        if (
            token_budget is not None
            and total_tokens >= token_budget
        ):
            result["status"] = "budget_exhausted"
            results.append(result)
            continue

        classification = classify_ticket(
            ticket_id,
            ticket["redacted_text"],
            system_prompt
        )

        result.update(classification)

        total_tokens += classification["tokens_used"]
        total_retries += classification["retry_count"]

        results.append(result)

    results_df = pd.DataFrame(results)

    return results_df, total_tokens, total_retries


zero_shot_results, zero_shot_tokens, zero_shot_retries = run_triage(
    zero_shot_prompt
)

print("Zero-shot classification completed.")
print("\nStatus counts:")
print(zero_shot_results["status"].value_counts())

print("\nTotal model tokens:", zero_shot_tokens)
print("Total recovered retries:", zero_shot_retries)

print(
    "Classified tickets:",
    zero_shot_results["status"].isin(
        ["classified_json", "classified_keyword"]
    ).sum()
)

Ticket T0002 | Retry 1/3 | Waiting 0.0493 seconds
Ticket T0116 | Retry 1/3 | Waiting 0.0504 seconds
Ticket T0153 | Retry 1/3 | Waiting 0.0533 seconds
Ticket T0039 | Retry 1/3 | Waiting 0.0455 seconds
Ticket T0152 | Retry 1/3 | Waiting 0.0486 seconds
Ticket T0043 | Retry 1/3 | Waiting 0.0554 seconds
Ticket T0111 | Retry 1/3 | Waiting 0.0478 seconds
Ticket T0155 | Retry 1/3 | Waiting 0.0458 seconds
Ticket T0100 | Retry 1/3 | Waiting 0.0499 seconds
Ticket T0068 | Retry 1/3 | Waiting 0.0481 seconds
Ticket T0135 | Retry 1/3 | Waiting 0.0456 seconds
Ticket T0009 | Retry 1/3 | Waiting 0.0455 seconds
Ticket T0067 | Retry 1/3 | Waiting 0.0534 seconds
Ticket T0106 | Retry 1/3 | Waiting 0.0469 seconds
Ticket T0097 | Retry 1/3 | Waiting 0.0478 seconds
Ticket T0003 | Retry 1/3 | Waiting 0.0504 seconds
Ticket T0019 | Retry 1/3 | Waiting 0.0539 seconds
Ticket T0086 | Retry 1/3 | Waiting 0.0494 seconds
Ticket T0065 | Retry 1/3 | Waiting 0.0537 seconds
Zero-shot classification completed.

Status counts

### Execute Few-Shot Classification

Run the same complete triage pipeline using the few-shot prompt.

The mock LLM uses the labelled examples in the system prompt to change its classification priority. Compare its results with the zero-shot run to evaluate the effect of examples on classification accuracy and token consumption.

In [12]:
few_shot_results, few_shot_tokens, few_shot_retries = run_triage(
    few_shot_prompt
)

print("Few-shot classification completed.")

print("\nStatus counts:")
print(few_shot_results["status"].value_counts())

print("\nTotal model tokens:", few_shot_tokens)
print("Total recovered retries:", few_shot_retries)

print(
    "Classified tickets:",
    few_shot_results["status"].isin(
        ["classified_json", "classified_keyword"]
    ).sum()
)

Ticket T0002 | Retry 1/3 | Waiting 0.0500 seconds
Ticket T0116 | Retry 1/3 | Waiting 0.0567 seconds
Ticket T0153 | Retry 1/3 | Waiting 0.0481 seconds
Ticket T0039 | Retry 1/3 | Waiting 0.0542 seconds
Ticket T0152 | Retry 1/3 | Waiting 0.0512 seconds
Ticket T0043 | Retry 1/3 | Waiting 0.0553 seconds
Ticket T0111 | Retry 1/3 | Waiting 0.0454 seconds
Ticket T0155 | Retry 1/3 | Waiting 0.0459 seconds
Ticket T0100 | Retry 1/3 | Waiting 0.0502 seconds
Ticket T0068 | Retry 1/3 | Waiting 0.0536 seconds
Ticket T0135 | Retry 1/3 | Waiting 0.0454 seconds
Ticket T0009 | Retry 1/3 | Waiting 0.0544 seconds
Ticket T0067 | Retry 1/3 | Waiting 0.0476 seconds
Ticket T0106 | Retry 1/3 | Waiting 0.0483 seconds
Ticket T0097 | Retry 1/3 | Waiting 0.0456 seconds
Ticket T0003 | Retry 1/3 | Waiting 0.0465 seconds
Ticket T0019 | Retry 1/3 | Waiting 0.0452 seconds
Ticket T0086 | Retry 1/3 | Waiting 0.0543 seconds
Ticket T0065 | Retry 1/3 | Waiting 0.0530 seconds
Few-shot classification completed.

Status counts:

### Compare Zero-Shot and Few-Shot Performance

Evaluate classification accuracy using only tickets with accepted classifications (`classified_json` and `classified_keyword`).

Compare:
- Accuracy against the original `true_category`.
- Total tokens consumed.
- Average tokens per classified ticket.
- Zero-shot confusion matrix.

The comparison will help determine whether the additional token cost of few-shot prompting is justified by its accuracy improvement.

In [13]:
def evaluate_classification(results, total_tokens):
    classified = results[
        results["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ].copy()

    accuracy = (
        classified["category"] == classified["true_category"]
    ).mean()

    return {
        "classified_tickets": len(classified),
        "accuracy": accuracy,
        "total_tokens": total_tokens,
        "tokens_per_classified": total_tokens / len(classified)
    }


zero_metrics = evaluate_classification(
    zero_shot_results, zero_shot_tokens
)

few_metrics = evaluate_classification(
    few_shot_results, few_shot_tokens
)

comparison = pd.DataFrame(
    [zero_metrics, few_metrics],
    index=["Zero-shot", "Few-shot"]
)

print("Classification performance comparison:")
display(comparison.round(3))

zero_classified = zero_shot_results[
    zero_shot_results["status"].isin(
        ["classified_json", "classified_keyword"]
    )
]

confusion_matrix = pd.crosstab(
    zero_classified["true_category"],
    zero_classified["category"],
    rownames=["Actual"],
    colnames=["Predicted"],
    dropna=False
).reindex(
    index=CATEGORIES,
    columns=CATEGORIES,
    fill_value=0
)

print("\nZero-shot confusion matrix:")
display(confusion_matrix)

print(
    "\nAdditional few-shot tokens:",
    few_shot_tokens - zero_shot_tokens
)

print(
    "Accuracy improvement (percentage points):",
    round(
        (few_metrics["accuracy"] - zero_metrics["accuracy"]) * 100,
        2
    )
)

Classification performance comparison:


,classified_tickets,accuracy,total_tokens,tokens_per_classified
Zero-shot,147,0.857,12364,84.109
Few-shot,147,1.000,21724,147.782



Zero-shot confusion matrix:


Predicted,billing,delivery,technical,fraud
Actual,,,,
billing,38,0,0,0
delivery,0,36,0,0
technical,10,0,29,0
fraud,11,0,0,23



Additional few-shot tokens: 9360
Accuracy improvement (percentage points): 14.29


Few-shot prompting is worth the additional cost in this experiment. It increased accuracy from 85.7% to 100%, correcting all 21 zero-shot classification errors. However, token consumption increased from 12,364 to 21,724, approximately 75.7% more, and the average cost increased from 84.11 to 147.78 tokens per classified ticket. For a support system where incorrect routing is costly, the accuracy improvement justifies the additional token usage.

### Enforce a Model Token Budget

Introduce a token budget that is checked before every model call.

For this demonstration, use the zero-shot prompt with a budget of 6,000 tokens. Once accumulated token usage reaches or exceeds the budget, stop making model calls and mark the remaining eligible tickets as `budget_exhausted`.

Save the results to `outputs/triage_budget_demo.csv`.

In [14]:
TOKEN_BUDGET = 6000

budget_results, budget_tokens, budget_retries = run_triage(
    zero_shot_prompt,
    token_budget=TOKEN_BUDGET
)

budget_output_path = OUTPUTS / "triage_budget_demo.csv"

budget_results.to_csv(budget_output_path, index=False)

print("Token budget:", TOKEN_BUDGET)
print("Tokens consumed:", budget_tokens)

print("\nBudget demonstration status counts:")
print(budget_results["status"].value_counts())

print(
    "\nBudget-exhausted tickets:",
    (budget_results["status"] == "budget_exhausted").sum()
)

print("\nSaved:", budget_output_path)

Ticket T0002 | Retry 1/3 | Waiting 0.0459 seconds


Ticket T0116 | Retry 1/3 | Waiting 0.0498 seconds
Ticket T0153 | Retry 1/3 | Waiting 0.0569 seconds
Ticket T0039 | Retry 1/3 | Waiting 0.0539 seconds
Ticket T0152 | Retry 1/3 | Waiting 0.0527 seconds
Ticket T0043 | Retry 1/3 | Waiting 0.0552 seconds
Ticket T0111 | Retry 1/3 | Waiting 0.0516 seconds
Ticket T0155 | Retry 1/3 | Waiting 0.0452 seconds
Ticket T0100 | Retry 1/3 | Waiting 0.0525 seconds
Token budget: 6000
Tokens consumed: 6038

Budget demonstration status counts:
status
budget_exhausted          80
classified_json           67
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     4
Name: count, dtype: int64

Budget-exhausted tickets: 80

Saved: d:\Capstone Project\Intelligence & Support Copilot — Capstone Project\Intelligence & Support Copilot\outputs\triage_budget_demo.csv


### Generate Final Few-Shot Triage Results

Run the few-shot classification pipeline with a token budget of 100,000.

Join the classification results with customer segments from Part B and apply the required routing rules:

- Guardrail failures, invalid outputs, and budget exhaustion → `human_review`, NORMAL.
- Fraud → `security_team`, HIGH.
- Premium customers → `premium_desk`, HIGH.
- All other classified tickets → `support_queue`, NORMAL.

Save the final 164-ticket result with the required output columns.

In [15]:
FINAL_TOKEN_BUDGET = 100000

final_results, final_tokens, final_retries = run_triage(
    few_shot_prompt,
    token_budget=FINAL_TOKEN_BUDGET
)

# Attach customer segment information from Part B.
segment_lookup = customer_segments[
    ["customer_id", "segment_name", "is_premium"]
].copy()

final_results = final_results.merge(
    segment_lookup,
    on="customer_id",
    how="left"
)

# Treat missing segment information as non-premium.
final_results["is_premium"] = (
    final_results["is_premium"]
    .astype("string")
    .str.lower()
    .eq("true")
)

def assign_routing(row):
    if row["status"] in [
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ]:
        return "human_review", "NORMAL"

    if row["category"] == "fraud":
        return "security_team", "HIGH"

    if row["is_premium"]:
        return "premium_desk", "HIGH"

    return "support_queue", "NORMAL"


routing = final_results.apply(assign_routing, axis=1)

final_results["route"] = routing.apply(lambda x: x[0])
final_results["priority"] = routing.apply(lambda x: x[1])

final_output = final_results[
    [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used"
    ]
].copy()

final_output[["category", "segment_name"]] = (
    final_output[["category", "segment_name"]].fillna("")
)

final_output_path = OUTPUTS / "triage_results.csv"
final_output.to_csv(final_output_path, index=False)

print("Final triage completed.")
print("Total tickets:", len(final_output))
print("Total tokens:", final_tokens)
print("Recovered retries:", final_retries)

print("\nStatus counts:")
print(final_output["status"].value_counts())

print("\nRouting counts:")
print(final_output["route"].value_counts())

print("\nPriority counts:")
print(final_output["priority"].value_counts())

print("\nSaved:", final_output_path)

Ticket T0002 | Retry 1/3 | Waiting 0.0536 seconds
Ticket T0116 | Retry 1/3 | Waiting 0.0484 seconds
Ticket T0153 | Retry 1/3 | Waiting 0.0555 seconds
Ticket T0039 | Retry 1/3 | Waiting 0.0532 seconds
Ticket T0152 | Retry 1/3 | Waiting 0.0538 seconds
Ticket T0043 | Retry 1/3 | Waiting 0.0552 seconds
Ticket T0111 | Retry 1/3 | Waiting 0.0495 seconds
Ticket T0155 | Retry 1/3 | Waiting 0.0469 seconds
Ticket T0100 | Retry 1/3 | Waiting 0.0459 seconds
Ticket T0068 | Retry 1/3 | Waiting 0.0473 seconds
Ticket T0135 | Retry 1/3 | Waiting 0.0521 seconds
Ticket T0009 | Retry 1/3 | Waiting 0.0479 seconds
Ticket T0067 | Retry 1/3 | Waiting 0.0471 seconds
Ticket T0106 | Retry 1/3 | Waiting 0.0538 seconds
Ticket T0097 | Retry 1/3 | Waiting 0.0471 seconds
Ticket T0003 | Retry 1/3 | Waiting 0.0539 seconds
Ticket T0019 | Retry 1/3 | Waiting 0.0563 seconds
Ticket T0086 | Retry 1/3 | Waiting 0.0454 seconds
Ticket T0065 | Retry 1/3 | Waiting 0.0496 seconds
Final triage completed.
Total tickets: 164
Total t

### Validate Final Triage Output

Perform final checks against the Part C acceptance criteria.

Verify:
- Exactly 164 tickets and the required eight output columns.
- All expected status counts.
- Every route and priority follows the specified business rules.
- Empty and long tickets have no classification or token usage.
- Human-review tickets have NORMAL priority.
- The final CSV can be reloaded successfully.

In [17]:
EXPECTED_COLUMNS = [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

# Reload the saved submission file.
saved_results = pd.read_csv(
    final_output_path,
    keep_default_na=False
)

# Check row count and schema.
assert len(saved_results) == 164
assert saved_results.columns.tolist() == EXPECTED_COLUMNS
assert saved_results["ticket_id"].is_unique

# Check expected status counts.
expected_status_counts = {
    "classified_json": 135,
    "classified_keyword": 12,
    "flagged_invalid_output": 9,
    "skipped_empty": 4,
    "flagged_long": 4
}

assert saved_results["status"].value_counts().to_dict() == expected_status_counts

# Check human-review routing.
human_statuses = [
    "skipped_empty",
    "flagged_long",
    "flagged_invalid_output",
    "budget_exhausted"
]

human_mask = saved_results["status"].isin(human_statuses)

assert (
    saved_results.loc[human_mask, "route"] == "human_review"
).all()

assert (
    saved_results.loc[human_mask, "priority"] == "NORMAL"
).all()

# Check fraud routing.
fraud_mask = (
    ~human_mask
    & saved_results["category"].eq("fraud")
)

assert (
    saved_results.loc[fraud_mask, "route"] == "security_team"
).all()

assert (
    saved_results.loc[fraud_mask, "priority"] == "HIGH"
).all()

# Check premium routing for non-fraud classified tickets.
premium_mask = (
    ~human_mask
    & ~fraud_mask
    & final_results["is_premium"].reset_index(drop=True)
)

assert (
    saved_results.loc[premium_mask, "route"] == "premium_desk"
).all()

assert (
    saved_results.loc[premium_mask, "priority"] == "HIGH"
).all()

# Check remaining classified tickets.
support_mask = (
    ~human_mask
    & ~fraud_mask
    & ~premium_mask
)

assert (
    saved_results.loc[support_mask, "route"] == "support_queue"
).all()

assert (
    saved_results.loc[support_mask, "priority"] == "NORMAL"
).all()

# Empty and long tickets must not have been sent to the model.
guardrail_mask = saved_results["status"].isin(
    ["skipped_empty", "flagged_long"]
)

assert (saved_results.loc[guardrail_mask, "tokens_used"] == 0).all()
assert saved_results.loc[guardrail_mask, "category"].eq("").all()

# Invalid responses must not have an assigned category.
invalid_mask = saved_results["status"].eq("flagged_invalid_output")

assert saved_results.loc[invalid_mask, "category"].eq("").all()

print("Final output validation: ALL CHECKS PASSED")
print("Rows:", len(saved_results))
print("Columns:", len(saved_results.columns))
print("Recovered retries:", final_retries)
print("Final tokens:", final_tokens)

Final output validation: ALL CHECKS PASSED
Rows: 164
Columns: 8
Recovered retries: 19
Final tokens: 21724


## Part C — Results and Findings

### 1. Input Processing and Privacy
- Total support tickets: 164
- Empty tickets skipped: 4
- Long tickets flagged: 4
- Eligible tickets classified: 156
- PII substitutions: 278
- Remaining PII matches: 0

### 2. Classification Performance

| Metric | Zero-Shot | Few-Shot |
|---|---:|---:|
| Classified tickets | 147 | 147 |
| Accuracy | 85.7% | 100% |
| Total tokens | 12,364 | 21,724 |
| Tokens per classified ticket | 84.11 | 147.78 |

Few-shot prompting improved accuracy by 14.3 percentage points, correcting all 21 zero-shot classification errors. It required approximately 75.7% more tokens. For a support system where incorrect routing is costly, the accuracy improvement justifies the additional token usage in this experiment.

### 3. Reliability and Budget
- Transient errors recovered: 19
- Maximum retries per ticket: 3
- Demonstration token budget: 6,000
- Tickets stopped by the budget: 80
- Final run token budget: 100,000

### 4. Final Ticket Routing

| Route | Tickets |
|---|---:|
| Support queue | 102 |
| Security team | 34 |
| Human review | 17 |
| Premium desk | 11 |
| **Total** | **164** |

### 5. Output Files
- `outputs/triage_budget_demo.csv`
- `outputs/triage_results.csv`

### 6. Validation and Human Review

The final output passed all validation checks.

JSON and keyword classifications were accepted after category validation. Empty tickets, long tickets, invalid model outputs, and budget-exhausted tickets are assigned to human review. Accepted classifications are rule-validated outputs, not independently verified human decisions.

In [2]:
print("FINAL TRIAGE INTEGRITY")
print("=" * 45)

checks = {
    "164 unique ticket IDs": (
        len(triage) == 164 and triage["ticket_id"].is_unique
    ),
    "No missing ticket IDs": triage["ticket_id"].notna().all(),
    "No missing customer IDs": triage["customer_id"].notna().all(),
    "No missing status": triage["status"].notna().all(),
    "No missing route": triage["route"].notna().all(),
    "Only valid priorities": set(triage["priority"].unique()) <= {"NORMAL", "HIGH"},
    "Budget demo has 164 tickets": len(budget) == 164,
    "Budget demo has unique ticket IDs": budget["ticket_id"].is_unique,
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

print("\nFinal triage routing by category:")
print(pd.crosstab(triage["category"], triage["route"]).to_string())

FINAL TRIAGE INTEGRITY
[PASS] 164 unique ticket IDs
[PASS] No missing ticket IDs
[PASS] No missing customer IDs
[PASS] No missing status
[PASS] No missing route
[PASS] Only valid priorities
[PASS] Budget demo has 164 tickets
[PASS] Budget demo has unique ticket IDs

Final triage routing by category:
route      premium_desk  security_team  support_queue
category                                             
billing               4              0             34
delivery              3              0             33
fraud                 0             34              0
technical             4              0             35
